In [ ]:
from llm.tensor import Tensor


t = Tensor([1,2,3])

print(t, t.data, t.data.dtype)

In [2]:
from llm.tensor import Tensor
import numpy as np

a = Tensor(2.0)
b = Tensor(3.0)
c = a + b

print(c.data, c.op, c._parents == (a, b))

c.grad = np.array(1.0)

c._backward()

print(a.grad, b.grad)

5.0 + True
1.0 1.0


In [1]:
from llm.tensor import Tensor

a,b,c = Tensor(2.0), Tensor(-3.0), Tensor(10.0)

d = a * b
e = d + c
L = e * e
L.backward()
print(L.data, a.grad, b.grad, c.grad)

16.0 -24.0 16.0 8.0


In [1]:
import numpy as np
from llm.tensor import Tensor


def f(x):
    return ((x * 2.0).tanh() + 1.0).log() * x.exp() - (x ** 2) / 3.0


x = Tensor(np.float64(0.7))
f(x).backward()

eps = 1e-6
up = f(Tensor(np.float64(0.7 + eps))).data
down = f(Tensor(0.7 - eps)).data

print(x.grad, (up - down ) / (2 * eps))

1.2720296887738574 1.319432584412361


In [2]:
import numpy as np
from llm.tensor import Tensor
x = Tensor(np.float64(0.7))
y = ((x * 2.0).tanh() + 1.0).log() * x.exp()
step = y
print(step.op, "<-", [p.op or "input" for p in step._parents])
step = step._parents[0]
print(step.op, "<-", [p.op or "input" for p in step._parents])

* <- ['log', 'exp']
log <- ['+']


In [3]:
x = Tensor(np.array([1.0, 2.0, 3.0]))
y = x * x
y.backward()

print(y.data, x.grad)

[1. 4. 9.] [2. 4. 6.]


In [2]:
from llm.tensor import Tensor, numerical_grad
import numpy as np
X = Tensor(np.array([[1., 2., 3.], [4., 5., 6.]]))
b = Tensor(np.array([10., 20., 30.]))
W = np.array([[1., 0., 2.],[3., 1., 1.]])

Y = (X + b) * Tensor(W)

Y.backward()

print(b.grad)
print(numerical_grad(lambda: ((X.data + b.data) * W).sum(), b.data))

[4. 1. 3.]
[3.99999999 1.         2.99999999]


In [1]:
import numpy as np
from llm.tensor import unbroadcast


G = np.array([[1., 0., 2.],[3., 1., 1.]])
print(unbroadcast(G, (3,)))
print(unbroadcast(G, (2,1)))
print(unbroadcast(G, (2,3)))

[4. 1. 3.]
[[3.]
 [5.]]
[[1. 0. 2.]
 [3. 1. 1.]]


In [1]:
import numpy as np

from llm.tensor import Tensor, numerical_grad


x = Tensor(np.array([[1., 2., 3.], [4., 5., 6.]]))
w = np.array([[1.], [-2.]])
(x.mean(axis=1) * Tensor(w[:, 0])).sum().backward()
print(x.grad)
f = lambda: (x.data.mean(axis=1) * w[:, 0]).sum()
print(numerical_grad(f, x.data))

[[ 0.33333333  0.33333333  0.33333333]
 [-0.66666667 -0.66666667 -0.66666667]]
[[ 0.33333333  0.33333333  0.33333333]
 [-0.66666667 -0.66666667 -0.66666667]]


In [1]:
import numpy as np
from llm.tensor import Tensor, numerical_grad


rng = np.random.default_rng(1)

A = Tensor(rng.normal(size=(2,3,4)))
B = Tensor(rng.normal(size=(4, 5)))
G = rng.normal(size=(2, 3, 5))

( A @ B * Tensor(G)).sum().backward()
f = lambda: (( A.data @ B.data) * G).sum()
print(np.abs(A.grad - numerical_grad(f, A.data)).max())
print(np.abs(B.grad - numerical_grad(f, B.data)).max())

4.840510214876304e-10
5.044689110889067e-10


In [3]:
rng = np.random.default_rng(0)

X  = Tensor(rng.normal(size=(4, 3)))
W = Tensor(rng.normal(size=(3, 5)))
b = Tensor(rng.normal(size=(5,)))


def loss_fn():
    h = (X @ W + b).tanh()
    return ((h ** 2).sum(axis=1) + 1.0).log().mean()

loss_fn().backward()
for name, t in [("X", X), ("W", W), ("b", b)]:
    num = numerical_grad(lambda: float(loss_fn().data), t.data)
    print(name, np.abs(t.grad - num).max()) 

X 1.0915632633889949e-10
W 1.2134603044611225e-10
b 1.9402279782809728e-10


In [6]:
x = np.array([[1.0, 2.0, 0.5],
              [0.0, 1.0, -1.0]])

W = np.array([[0.5, 1.0],
              [1.0, 0.0],
              [-0.5, 2.0]])
b = np.array([0.1, -0.2])

print(x @ W + b)

[[ 2.35  1.8 ]
 [ 1.6  -2.2 ]]
